In [97]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
import numpy as np

from model import WeatherLSTM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PATH = 'data/clean_data.csv'
DATE = '2017-03-18'

In [98]:
def predict_for_specific_day(model, df, target_date, seq_length=7):
    model.eval()
    predictions = []
    
    df['Date'] = pd.to_datetime(df['Date'])
    target_date = pd.to_datetime(target_date)

    for city, group in df.groupby('Location'):
        group = group.sort_values('Date')
        target_idx = group.index[group['Date'] == target_date]
        
        if len(target_idx) > 0:
            idx = group.index.get_loc(target_idx[0])
            
            if idx >= seq_length:
                seq_data = group.iloc[idx-seq_length:idx][features_cols].values
                
                input_tensor = torch.tensor(seq_data, dtype=torch.float32).unsqueeze(0).to(device)
                input_tensor = input_tensor.permute(0, 2, 1)
                
                with torch.no_grad():
                    pred_temp, pred_rain_logits = model(input_tensor)
                    
                    prob_rain = torch.sigmoid(pred_rain_logits).item()
                    
                val_norm = pred_temp.item()
                temp_reelle = (val_norm * std_temp) + mean_temp
                
                predictions.append({
                    'Ville': city,
                    'Date_Predite': target_date,
                    'Temp_Predite': round(temp_reelle, 2), # Maintenant round() fonctionne sur un float
                    'Prob_Pluie': round(prob_rain, 4),
                    'Pluie_Prevue': 1 if prob_rain > 0.5 else 0
                })

    results_df = pd.DataFrame(predictions)
    
    results_df.to_csv(f"predict/predict_{target_date.date()}.csv", index=False)
    return results_df

In [99]:
checkpoint = torch.load('weather_model.pth', map_location=device)

scaler = checkpoint['scaler']
features_cols = checkpoint['features']
le_city = checkpoint['city_encoder']

num_features = len(features_cols)
model = WeatherCNN(num_features=num_features).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

df_map = predict_for_specific_day(model, df, DATE)

In [100]:
def evaluate_model_performance(model, df, target_date, seq_length=7):
    model.eval()
    results = []
    
    target_dt = pd.to_datetime(target_date)

    for city, group in df.groupby('Location'):
        group = group.sort_values('Date')
        target_row = group[group['Date'] == target_dt]
        
        if not target_row.empty:
            idx = group.index.get_loc(target_row.index[0])
            
            if idx >= seq_length:
                seq_data = group.iloc[idx-seq_length:idx][features_cols].values
                input_tensor = torch.tensor(seq_data, dtype=torch.float32).unsqueeze(0).to(device)
                input_tensor = input_tensor.permute(0, 2, 1)
                
                with torch.no_grad():
                    pred_temp, pred_rain_logits = model(input_tensor)
                    prob_rain = torch.sigmoid(pred_rain_logits).item()

                actual_temp = target_row['MaxTemp'].values[0]
                actual_rain = target_row['RainToday'].values[0]

                results.append({
                    'Erreur_Temp': abs(pred_temp.item() - actual_temp),
                    'Succes_Pluie': int((prob_rain > 0.5) == actual_rain)
                })

    res_df = pd.DataFrame(results)
    
    mean_temp_error = res_df['Erreur_Temp'].mean()
    accuracy_rain = res_df['Succes_Pluie'].mean()

    print(f"--- Performances au {target_date} ---")
    print(f"Écart de température moyen : {mean_temp_error:.2f}°C")
    print(f"Accuracy Pluie (Précision) : {accuracy_rain * 100:.1f}%")
    
    return mean_temp_error, accuracy_rain

ecart_moyen, precision = evaluate_model_performance(model, df, DATE)

--- Performances au 2017-03-18 ---
Écart de température moyen : 4.91°C
Accuracy Pluie (Précision) : 65.3%
